<a href="https://colab.research.google.com/github/priyaking3803-tech/Business-Reputation-Insights-Analyzer-using-Google-Maps-Reviews-/blob/main/Copy_of_Business_Reputation_%26_Insights_Analyzer_using_Google_Maps_Reviews_%2B_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#  Google Maps Reviews Fetcher (SerpAPI Edition)
import requests, pandas as pd

# 1. Set parameters
params = {
    "engine": "google_maps_reviews",
    "place_id": "ChIJLU7jZClu5kcR4PcOOO6p3I0",  # Example: Louvre Museum, Paris
    "hl": "en",
    "api_key":xxx  # 🔑 Replace with your key
}

#  2. Fetch data
response = requests.get("https://serpapi.com/search.json", params=params)
reviews = response.json().get("reviews", [])

#  3. Structure the data
df = pd.DataFrame([{
    "user": r.get("user"),
    "rating": r.get("rating"),
    "date": r.get("date"),
    "review": r.get("snippet")
} for r in reviews])

#  4. Save output
df.to_csv("google_reviews.csv", index=False)
print(" Done! Reviews saved to google_reviews.csv")
print(df.head(3))


 Done! Reviews saved to google_reviews.csv
                                                user  rating         date  \
0  {'name': 'Diego Onomátor Rodriguez', 'link': '...     5.0   5 days ago   
1  {'name': 'David Alpern', 'link': 'https://www....     5.0  a month ago   
2  {'name': 'Константин Новков', 'link': 'https:/...     5.0  a month ago   

                                              review  
0  This is an iconic landmark in Paris so there i...  
1  I visited Paris about 10 years ago with only a...  
2  I get why it’s the most popular landmark. The ...  


In [ ]:
#  Google Reviews Text Preprocessing
import pandas as pd
import re

# 1️ Load your dataset
df = pd.read_csv("google_reviews.csv")

# 2️ Basic text cleaning function
def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)        # remove links
    text = re.sub(r"[^A-Za-z0-9\s.,!?]", "", text)    # remove emojis/symbols
    text = re.sub(r"\s+", " ", text).strip()          # clean spaces
    return text.lower()                               # lowercase for NLP

# 3️ Apply cleaning
df["clean_review"] = df["review"].apply(clean_text)

# 4️ Optional: remove duplicates / missing text
df = df.drop_duplicates(subset="clean_review")
df = df[df["clean_review"].str.strip() != ""]

# 5️ Save structured & cleaned dataset
df.to_csv("cleaned_reviews.csv", index=False)
print(" Cleaned reviews saved to cleaned_reviews.csv")
print(df.head(3))


 Cleaned reviews saved to cleaned_reviews.csv
                                                user  rating         date  \
0  {'name': 'Diego Onomátor Rodriguez', 'link': '...     5.0   5 days ago   
1  {'name': 'David Alpern', 'link': 'https://www....     5.0  a month ago   
2  {'name': 'Константин Новков', 'link': 'https:/...     5.0  a month ago   

                                              review  \
0  This is an iconic landmark in Paris so there i...   
1  I visited Paris about 10 years ago with only a...   
2  I get why it’s the most popular landmark. The ...   

                                        clean_review  
0  this is an iconic landmark in paris so there i...  
1  i visited paris about 10 years ago with only a...  
2  i get why its the most popular landmark. the v...  


In [ ]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 1.8 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade langchain langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.10
    Uninstalling langgraph-1.0.10:
      Successfully uninstalled langgraph-1.0.10
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.10
    Uninstalling langchain-1.2.10:
      Successfully uninstalled langchain-1.2.10
ERROR: pip's dependency resolver

In [ ]:
import pandas as pd

from langchain_community.llms import HuggingFacePipeline

In [ ]:
import pandas as pd
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# --- 1. LLM SETUP (Hugging Face Model) ---
# We will use a fast, instruction-following T5 model to perform extraction and summarization.
# Note: This will download the model weights (a few hundred MB) the first time it runs.
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a Hugging Face Pipeline for Text2Text Generation
# The task 'text2text-generation' was not recognized previously, indicating a potential environment issue.
# We will revert to 'text-generation' as it is an available task in the current transformers version.
pipe = pipeline(
    "text-generation", # Changed back to "text-generation" as per available tasks
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    device=-1, # Use -1 for CPU, or 0 for GPU
    torch_dtype=torch.float32 # Use torch.bfloat16 for faster inference on newer CPUs/GPUs
)

# Wrap the pipeline in LangChain's HuggingFacePipeline
hf_llm = HuggingFacePipeline(pipeline=pipe)

# --- 2. Sentiment & Topic Extraction (Prompt-driven) ---
review_template = """
Analyze the following Google Maps review and extract its sentiment (positive, neutral, negative) and all relevant topics (service, pricing, ambience, staff, food/product quality, speed).
Return the result in the exact format: SENTIMENT: [sentiment] | TOPICS: [topic1, topic2, ...]

Review: "{review}"
"""
prompt_extraction = PromptTemplate(
    template=review_template,
    input_variables=["review"]
)
extraction_chain = prompt_extraction | hf_llm

single_review = "The staff was rude and the waiting time for a simple coffee was 20 minutes, but the seating area was comfortable."
# Uncomment to test extraction (this will use the T5 model)
# result_extraction = extraction_chain.invoke({"review": single_review})
# print("--- 1. Extraction Result (Hugging Face) ---")
# print(result_extraction)
# Expected: SENTIMENT: negative | TOPICS: [staff, speed, ambience]

# --- 3. Summarized Insights (Weekly/Monthly Highlights) ---
weekly_insights = [
    "40% of negative reviews mention 'staff' or 'service'.",
    "The 'food/product quality' theme was overwhelmingly positive (95% score).",
    "The average rating fell from 4.2 to 3.5 this week.",
    "The top complaint theme was 'speed' of service.",
]
input_context = "\n- " + "\n- ".join(weekly_insights)

summary_template = """
You are a business intelligence analyst. Review the following customer feedback summary from the past week.
Provide a professional summary with three key points: one strength, one urgent concern, and an overall conclusion.

Context:
{context}
"""

summary_prompt = PromptTemplate(template=summary_template, input_variables=["context"])
summary_chain = summary_prompt | hf_llm

# Uncomment to test summarization
# summary_result = summary_chain.invoke({"context": input_context})
# print("\n--- 2. Summarized Insights (Hugging Face) ---")
# print(summary_result)

# --- 4. Recommendation Generation ---
top_negative_themes = [
    "Staff attitude was a major issue, contributing to 40% of negative reviews.",
    "Speed of service during 11 AM - 1 PM was consistently the top complaint this week.",
]

recommendation_template = """
You are an Operations Consultant. Given the key negative themes for a business, provide three concrete, actionable, and specific recommendations for operational improvement.

Top Negative Themes:
{themes}

Recommendations (Format as a numbered list):
"""

recommendation_prompt = PromptTemplate(template=recommendation_template, input_variables=["themes"])
recommendation_chain = recommendation_prompt | hf_llm

# Uncomment to test recommendations
# recommendation_result = recommendation_chain.invoke(
#     {
#         "themes": "\n- ".join(top_negative_themes)
#     }
# )
# print("\n--- 3. Recommendation Generation (Hugging Face) ---")
# print(recommendation_result)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2

In [ ]:
!pip install --upgrade transformers==4.41.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.6.0
    Uninstalling huggingface_hub-1.6.0:
      Successfully uninstalled huggingface_hub-1.6.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install langchain-community

In [ ]:
import pandas as pd
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer
import torch
import os # Import os to access environment variables
from langchain_openai import ChatOpenAI # Import ChatOpenAI

# --- 1. LLM SETUP (Hugging Face Model) ---
# We will use a fast, instruction-following T5 model to perform extraction and summarization.
# Note: This will download the model weights (a few hundred MB) the first time it runs.
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a Hugging Face Pipeline for Text2Text Generation
# The task 'text2text-generation' was not recognized previously, indicating a potential environment issue.
# Reverting to 'summarization' as it is a common and supported task for T5 models.
pipe = pipeline(
    "summarization", # Changed to "summarization" to resolve KeyError with "text2text-generation"
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    device=-1, # Use -1 for CPU, or 0 for GPU
    torch_dtype=torch.float32 # Use torch.bfloat16 for faster inference on newer CPUs/GPUs
)

# Wrap the pipeline in LangChain's HuggingFacePipeline
hf_llm = HuggingFacePipeline(pipeline=pipe)

# --- OR ---

# --- LLM SETUP (OpenAI Model) ---
# If you prefer to use an OpenAI model, uncomment the following lines.
# Ensure your OPENAI_API_KEY is set as an environment variable (e.g., using Colab Secrets and `os.environ`).
# You can use the cell I provided earlier (`eda6b556`) to load the API key from secrets.
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7) # Uncomment this line to use OpenAI


# --- 2. Sentiment & Topic Extraction (Prompt-driven) ---
review_template = """
Analyze the following Google Maps review and extract its sentiment (positive, neutral, negative) and all relevant topics (service, pricing, ambience, staff, food/product quality, speed).
Return the result in the exact format: SENTIMENT: [sentiment] | TOPICS: [topic1, topic2, ...]

Review: "{review}"
"""
prompt_extraction = PromptTemplate(
    template=review_template,
    input_variables=["review"]
)
# Use hf_llm for the extraction chain if using Hugging Face model
extraction_chain = prompt_extraction | hf_llm

single_review = "The staff was rude and the waiting time for a simple coffee was 20 minutes, but the seating area was comfortable."
# Uncomment to test extraction (this will use the T5 model)
# result_extraction = extraction_chain.invoke({"review": single_review})
# print("--- 1. Extraction Result (Hugging Face) ---")
# print(result_extraction)
# Expected: SENTIMENT: negative | TOPICS: [staff, speed, ambience]

# If you uncommented and are using the OpenAI model, you would use:
# extraction_chain_openai = prompt_extraction | llm
# result_extraction_openai = extraction_chain_openai.invoke({"review": single_review})
# print("--- 1. Extraction Result (OpenAI) ---")
# print(result_extraction_openai)


# --- 3. Summarized Insights (Weekly/Monthly Highlights) ---
weekly_insights = [
    "40% of negative reviews mention 'staff' or 'service'.",
    "The 'food/product quality' theme was overwhelmingly positive (95% score).",
    "The average rating fell from 4.2 to 3.5 this week.",
    "The top complaint theme was 'speed' of service.",
]
input_context = "\n- " + "\n- ".join(weekly_insights)

summary_template = """
You are a business intelligence analyst. Review the following customer feedback summary from the past week.
Provide a professional summary with three key points: one strength, one urgent concern, and an overall conclusion.

Context:
{context}
"""

summary_prompt = PromptTemplate(template=summary_template, input_variables=["context"])
# Use hf_llm for the summary chain if using Hugging Face model
summary_chain = summary_prompt | hf_llm

# Uncomment to test summarization
# summary_result = summary_chain.invoke({"context": input_context})
# print("\n--- 2. Summarized Insights (Hugging Face) ---")
# print(summary_result)

# If you uncommented and are using the OpenAI model, you would use:
# summary_chain_openai = summary_prompt | llm
# summary_result_openai = summary_chain_openai.invoke({"context": input_context})
# print("--- 2. Summarized Insights (OpenAI) ---")
# print(summary_result_openai)


# --- 4. Recommendation Generation ---
top_negative_themes = [
    "Staff attitude was a major issue, contributing to 40% of negative reviews.",
    "Speed of service during 11 AM - 1 PM was consistently the top complaint this week.",
]

recommendation_template = """
You are an Operations Consultant. Given the key negative themes for a business, provide three concrete, actionable, and specific recommendations for operational improvement.

Top Negative Themes:
{themes}

Recommendations (Format as a numbered list):
"""

recommendation_prompt = PromptTemplate(template=recommendation_template, input_variables=["themes"])
# Use hf_llm for the recommendation chain if using Hugging Face model
recommendation_chain = recommendation_prompt | hf_llm

# Uncomment to test recommendations
# recommendation_result = recommendation_chain.invoke({
#     "themes": "\n- ".join(top_negative_themes)
# })
# print("\n--- 3. Recommendation Generation (Hugging Face) ---")
# print(recommendation_result)

# If you uncommented and are using the OpenAI model, you would use:
# recommendation_chain_openai = recommendation_prompt | llm
# recommendation_result_openai = recommendation_chain_openai.invoke({
#     "themes": "\n- ".join(top_negative_themes)
# })
# print("--- 3. Recommendation Generation (OpenAI) ---")
# print(recommendation_result_openai)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/tmp/ipykernel_1270/924607689.py:29: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  hf_llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
!pip install langchain-community transformers torch

In [ ]:
!pip install streamlit streamlit_option_menu  # installing streamlit and streamlit_option_menu packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 29.6 MB/s eta 0:00:00


In [ ]:

# my file name is app.py you can change it to anything you want

%%writefile app.py


import streamlit as st
import requests
import uuid

# --- Langflow API URL (your flow endpoint)
url = "https://aws-us-east-2.langflow.datastax.com/lf/58d90fd2-67ed-4afa-9017-742cac8a98e8/api/v1/run/aef8f281-4f24-473b-8536-ef23d2a8b2cb?stream=false"

# --- Headers (update your application token)
headers = {
    "X-DataStax-Current-Org": "6e8e414b-573b-4a66-9839-a7d453c1669e",
    "Authorization": "Bearer xxx
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# --- Function to call Langflow API
def analyze_review(review_text):
    session_id = str(uuid.uuid4())

    payload = {
        "output_type": "chat",
        "input_type": "chat",
        "input_value": review_text,
        "session_id": session_id
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()

        # Extract AI response text
        if "outputs" in data and len(data["outputs"]) > 0:
            return data["outputs"][0]["outputs"][0]["results"]["message"]["text"]

        return "⚠️ No valid response from Langflow."

    except Exception as e:
        return f"❌ Error: {str(e)}"

# --- Streamlit UI
st.set_page_config(page_title="Business Reputation & Insights Analyzer", layout="wide")

st.title("📊 Business Reputation & Insights Analyzer")
st.write("Enter a Google Maps review and get AI-powered insights using DataStax Langflow.")

review_text = st.text_area("Paste Google Review Here:", height=150)

if st.button("Analyze Review"):
    if review_text.strip() == "":
        st.error("Please enter a review!")
    else:
        with st.spinner("Analyzing review..."):
            result = analyze_review(review_text)
        st.success("Analysis Complete!")
        st.text_area("AI Insights:", value=result, height=200)


Overwriting app.py


In [ ]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-03-13 08:12:03--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64 [following]
--2026-03-13 08:12:03--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/731ab2f8-6b77-4adb-a7b3-1104525e9d72?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-13T09%3A04%3A45Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-13T0

In [ ]:
!streamlit run /content/app.py &>/content/logs.txt &  # here instead of app.py please rename with your file name

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://position-consistency-pensions-essentially.trycloudflare.com


In [ ]:

# my file name is app.py you can change it to anything you want

%%writefile app.py


import streamlit as st
import requests
import uuid

# ---- Background Style
page_bg = """
<style>

/* Full page background */
.stApp {
background: linear-gradient(135deg,#FFF9C4,#C8E6C9);
}

/* Header transparent */
header{
background: transparent !important;
}

/* Text area style */
textarea{
background-color: white !important;
color: black !important;
border-radius:10px !important;
border:2px solid #81C784 !important;
}

/* Button style */
.stButton>button{
background-color:#2E7D32 !important;
color:white !important;
border-radius:12px !important;
padding:10px 20px !important;
font-size:16px !important;
border:none !important;
}

/* Title style */
h1{
color:#1B5E20 !important;
text-align:center;
}

/* Light dot pattern */
.stApp::before{
content:"";
position:fixed;
top:0;
left:0;
width:100%;
height:100%;
background-image: radial-gradient(#A5D6A7 1px, transparent 1px);
background-size:20px 20px;
opacity:0.2;
pointer-events:none;
}

</style>
"""

st.markdown(page_bg, unsafe_allow_html=True)

# --- Langflow API URL
url = "https://aws-us-east-2.langflow.datastax.com/lf/58d90fd2-67ed-4afa-9017-742cac8a98e8/api/v1/run/aef8f281-4f24-473b-8536-ef23d2a8b2cb?stream=false"

headers = {
    "X-DataStax-Current-Org": "6e8e414b-573b-4a66-9839-a7d453c1669e",
    "Authorization":  "Bearer AstraCSxxx
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# --- API Function
def analyze_review(review_text):
    session_id = str(uuid.uuid4())

    payload = {
        "output_type": "chat",
        "input_type": "chat",
        "input_value": review_text,
        "session_id": session_id
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()

        if "outputs" in data and len(data["outputs"]) > 0:
            return data["outputs"][0]["outputs"][0]["results"]["message"]["text"]

        return "No valid response"

    except Exception as e:
        return f"Error: {str(e)}"


# ---- UI
st.title("⭐ Google Review Analyzer")

st.write("Paste a Google Maps review and get AI insights.")

review_text = st.text_area("Paste Google Review Here", height=150)

if st.button("Analyze Review"):

    if review_text.strip() == "":
        st.error("Please enter a review")

    else:
        with st.spinner("Analyzing Review..."):
            result = analyze_review(review_text)

        st.success("Analysis Complete")

        st.text_area("AI Insights", value=result, height=200)

Overwriting app.py


In [ ]:

# my file name is app.py you can change it to anything you want

%%writefile app.py



import streamlit as st
import requests
import uuid

# --- Page Config
st.set_page_config(page_title="Business Reputation & Insights Analyzer", layout="wide")

# ---- Background Style
page_bg = """
<style>

/* Main background gradient */
[data-testid="stAppViewContainer"]{
background: linear-gradient(135deg,#FFF9C4,#C8E6C9);
background-attachment: fixed;
}

/* Header transparent */
[data-testid="stHeader"]{
background: rgba(0,0,0,0);
}

/* Text Area Style */
.stTextArea textarea{
background-color: #ffffff;
color: black;
border-radius:10px;
border:2px solid #81C784;
}

/* Button Style */
.stButton>button{
background-color: #2E7D32;
color: white;
font-size:16px;
border-radius:12px;
padding:10px 20px;
border:none;
}

/* Title Style */
h1{
color:#1B5E20;
text-align:center;
}

/* Pattern background effect */
[data-testid="stAppViewContainer"]::before{
content:"";
position:fixed;
top:0;
left:0;
width:100%;
height:100%;
background-image: radial-gradient(#A5D6A7 1px, transparent 1px);
background-size:20px 20px;
opacity:0.2;
pointer-events:none;
}

</style>
"""

# Apply background style
st.markdown(page_bg, unsafe_allow_html=True)


# --- Langflow API URL
url = "https://aws-us-east-2.langflow.datastax.com/lf/58d90fd2-67ed-4afa-9017-742cac8a98e8/api/v1/run/aef8f281-4f24-473b-8536-ef23d2a8b2cb?stream=false"

# --- Headers
headers = {
    "X-DataStax-Current-Org": "6e8e414b-573b-4a66-9839-a7d453c1669e",
    "Authorization":  "Bearer AstraCS:xxx
    "Content-Type": "application/json",
    "Accept": "application/json",
}

# --- Function to call Langflow API
def analyze_review(review_text):
    session_id = str(uuid.uuid4())

    payload = {
        "output_type": "chat",
        "input_type": "chat",
        "input_value": review_text,
        "session_id": session_id
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()

        if "outputs" in data and len(data["outputs"]) > 0:
            return data["outputs"][0]["outputs"][0]["results"]["message"]["text"]

        return "⚠️ No valid response from Langflow."

    except Exception as e:
        return f"❌ Error: {str(e)}"


# --- Streamlit UI
st.title("📊 Business Reputation & Insights Analyzer")
st.subheader("📍 Google Maps Reviews + 🤖 LLM Powered Insights")

st.write("Enter a Google Maps review and get AI-powered insights using DataStax Langflow.")

review_text = st.text_area("Paste Google Review Here:", height=150)

if st.button("Analyze Review"):
    if review_text.strip() == "":
        st.error("Please enter a review!")
    else:
        with st.spinner("Analyzing review..."):
            result = analyze_review(review_text)

        st.success("Analysis Complete!")
        st.text_area("AI Insights:", value=result, height=200)



Writing app.py
